In [1]:
import joblib, json, datetime
import numpy as np
import pandas as pd
import sklearn, s3fs

art = joblib.load("../artifacts/final_model.joblib")
model, FEATS, THRESHOLD, METRICS = art["model"], art["features"], art["threshold"], art["metrics"]

DATA = "s3://sagemaker-salesmesh-myra/processed/"
train = pd.read_csv(DATA + "train.csv")
test = pd.read_csv(DATA + "test.csv")

CAT = ["source", "device_os"]
NUM = [f for f in FEATS if f not in CAT]
print(len(FEATS), "features | threshold:", THRESHOLD, "| sklearn:", sklearn.__version__)

13 features | threshold: 0.035 | sklearn: 1.7.2


In [2]:
metadata = {
    "model_name": "salesmesh-fake-user-detector",
    "model_version": "v1",
    "model_type": "LogisticRegression (C=0.01, class_weight=balanced) + sigmoid calibration",
    "created_at": datetime.date.today().isoformat(),
    "training_data": "BAF Base (NeurIPS 2022), months 0-5, 794,989 rows",
    "test_data": "BAF Base, months 6-7, 205,011 rows",
    "sklearn_version": sklearn.__version__,
    "features": FEATS,
    "numeric_features": NUM,
    "categorical_features": {c: sorted(train[c].dropna().unique().tolist()) for c in CAT},
    "missing_value_rule": "Send null (None) for unknown values. Negative values are not valid.",
    "numeric_ranges": {c: {"min": float(train[c].min()), "max": float(train[c].max()),
                           "median": float(train[c].median())} for c in NUM},
    "threshold": float(THRESHOLD),
    "threshold_rule": "flag if probability >= threshold (5% FPR on test months)",
    "metrics_test": {k: float(v) for k, v in METRICS.items()},
    "excluded_features": {"customer_age": "fairness (FPR ratio 0.18 -> 0.47)",
                          "velocity_*, zip_count_4w": "bank-scale counts, no added value"},
}
print(json.dumps(metadata, indent=2)[:1500])

{
  "model_name": "salesmesh-fake-user-detector",
  "model_version": "v1",
  "model_type": "LogisticRegression (C=0.01, class_weight=balanced) + sigmoid calibration",
  "created_at": "2026-10-15",
  "training_data": "BAF Base (NeurIPS 2022), months 0-5, 794,989 rows",
  "test_data": "BAF Base, months 6-7, 205,011 rows",
  "sklearn_version": "1.7.2",
  "features": [
    "name_email_similarity",
    "prev_address_months_count",
    "current_address_months_count",
    "date_of_birth_distinct_emails_4w",
    "email_is_free",
    "phone_home_valid",
    "phone_mobile_valid",
    "foreign_request",
    "source",
    "session_length_in_minutes",
    "device_os",
    "keep_alive_session",
    "device_distinct_emails_8w"
  ],
  "numeric_features": [
    "name_email_similarity",
    "prev_address_months_count",
    "current_address_months_count",
    "date_of_birth_distinct_emails_4w",
    "email_is_free",
    "phone_home_valid",
    "phone_mobile_valid",
    "foreign_request",
    "session_leng

In [3]:
sample = pd.concat([test[test.fraud_bool == 1].sample(10, random_state=1),
                    test[test.fraud_bool == 0].sample(10, random_state=1)])
probs = model.predict_proba(sample[FEATS])[:, 1]
golden = [{"features": {k: (None if pd.isna(v) else (v.item() if hasattr(v, "item") else v))
                        for k, v in row[FEATS].items()},
           "expected_probability": round(float(p), 6),
           "expected_flag": bool(p >= THRESHOLD),
           "true_label": int(row["fraud_bool"])}
          for (_, row), p in zip(sample.iterrows(), probs)]

base = model.calibrated_classifiers_[0].estimator
names = base.named_steps["prep"].get_feature_names_out().tolist()
coefs = base.named_steps["model"].coef_[0].tolist()
coefficients = dict(zip(names, [round(c, 6) for c in coefs]))

print(len(golden), "golden examples |", sum(g["expected_flag"] for g in golden), "flagged")

20 golden examples | 7 flagged


In [4]:
import os
OUT = "../artifacts/export/v1"
os.makedirs(OUT, exist_ok=True)

joblib.dump(model, f"{OUT}/model.joblib")
for name, obj in [("metadata", metadata), ("golden_examples", golden), ("coefficients", coefficients)]:
    with open(f"{OUT}/{name}.json", "w") as f:
        json.dump(obj, f, indent=2)

fs = s3fs.S3FileSystem()
for fname in ["model.joblib", "metadata.json", "golden_examples.json", "coefficients.json"]:
    fs.put(f"{OUT}/{fname}", f"sagemaker-salesmesh-myra/models/final/v1/{fname}")
print(fs.ls("sagemaker-salesmesh-myra/models/final/v1/"))

['sagemaker-salesmesh-myra/models/final/v1/coefficients.json', 'sagemaker-salesmesh-myra/models/final/v1/golden_examples.json', 'sagemaker-salesmesh-myra/models/final/v1/metadata.json', 'sagemaker-salesmesh-myra/models/final/v1/model.joblib']


In [5]:
fs.get("sagemaker-salesmesh-myra/models/final/v1/model.joblib", "/tmp/check_model.joblib")
loaded = joblib.load("/tmp/check_model.joblib")

X = pd.DataFrame([g["features"] for g in golden])[FEATS]
new_probs = loaded.predict_proba(X)[:, 1]
expected = np.array([g["expected_probability"] for g in golden])
print("Max difference:", float(np.abs(new_probs - expected).max()))
print("PASS" if np.allclose(new_probs, expected, atol=1e-6) else "FAIL")

Max difference: 4.96062590097171e-07
PASS
